In [4]:
import tensorflow as tf
from tensorflow.keras import layers, Model

SRC_VOCAB = 10000
TGT_VOCAB = 10000
EMBED_DIM = 128
UNITS = 256

# Encoder
encoder_inputs = layers.Input(
    shape=(None,),
    name="encoder_input"
)

encoder_embedding = layers.Embedding(
    SRC_VOCAB,
    EMBED_DIM
)(encoder_inputs)

encoder_bilstm = layers.Bidirectional(
    layers.LSTM(
        UNITS,
        return_sequences=True,
        return_state=True
    )
)

encoder_output, fh, fc, bh, bc = encoder_bilstm(
    encoder_embedding
)

# Combine forward and backward states
state_h = layers.Concatenate()([fh, bh])
state_c = layers.Concatenate()([fc, bc])

# Decoder
decoder_inputs = layers.Input(
    shape=(None,),
    name="decoder_input"
)

decoder_embedding = layers.Embedding(
    TGT_VOCAB,
    EMBED_DIM
)(decoder_inputs)

decoder_lstm = layers.LSTM(
    UNITS * 2,
    return_sequences=True
)

decoder_output = decoder_lstm(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

outputs = layers.Dense(
    TGT_VOCAB,
    activation="softmax"
)(decoder_output)

model = Model(
    [encoder_inputs, decoder_inputs],
    outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │  1,280,000 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ [(None, None,     │    788,480 │ embedding[0][0]   │
│ (Bidirectional)     │ 512), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │  1,280,000 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, None, 512) │  1,312,768 │ embedding_1[0][0… │
│                     │                   │            │ concatenate[0][0… │
│                     │                   │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None,      │  5,130,000 │ lstm_1[0][0]      │
│                     │ 10000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,791,248 (37.35 MB)

 Trainable params: 9,791,248 (37.35 MB)

 Non-trainable params: 0 (0.00 B)